# Stress-testing estimator robustness

Research question: how much does an estimator move when a clean synthetic signal is contaminated by controlled level shifts?

This notebook uses stress-test mode to create clean and contaminated records from the same synthetic baseline.

In [ ]:
from pathlib import Path

import pandas as pd

from lrdbench.output_contract import validate_output_contract
from lrdbench.runner import run_manifest_mapping

In [ ]:
manifest = {
    "manifest_id": "workshop_stress_test_v1",
    "name": "workshop_stress_test",
    "mode": "stress_test",
    "source": {
        "type": "generator_grid",
        "generators": [
            {"family": "fGn", "params": {"H": [0.7], "n": [128], "sigma": [1.0]}, "replicates": 2}
        ],
    },
    "contamination": {
        "operators": [
            {"name": "level_shift", "params": {"shift": [0.15, 0.35]}}
        ]
    },
    "estimators": [
        {"name": "RS", "family": "time_domain", "target_estimand": "hurst_scaling_proxy", "supports_ci": False}
    ],
    "metrics": ["bias", "mae", "estimate_drift", "relative_degradation_ratio", "validity_rate", "runtime"],
    "leaderboards": [
        {
            "name": "robustness_summary",
            "mode": "stress_test",
            "component_metrics": ["estimate_drift", "validity_rate", "runtime"],
            "weights": {"estimate_drift": 0.5, "validity_rate": 0.3, "runtime": 0.2},
            "ranking_rule": "weighted_rank",
            "tie_break_rule": "estimate_drift",
        }
    ],
    "report": {"formats": ["html", "csv"], "export_root": "reports/notebooks/stress_test"},
    "seeds": {"global_seed": 20260427},
    "execution": {"max_workers": 1},
}

In [ ]:
out = run_manifest_mapping(manifest, base_dir=Path.cwd())
errors = validate_output_contract(out.result_store_path)
assert errors == []
out.result_store_path

In [ ]:
result_store = Path(out.result_store_path)
records = pd.read_csv(result_store / "raw" / "records.csv")
metrics = pd.read_csv(result_store / "tables" / "per_stratum_metrics.csv")
records[["record_id", "source_type", "annotations_json"]].head(), metrics[["metric_name", "value"]].head()